In [ ]:
print("Hybrid Search Demo - BM25 + Vector Search")

### Data Ingestion & Setup

In [2]:
from pathlib import Path
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from dotenv import load_dotenv

# Load file
file_path = Path("../data/information.txt").resolve()
loader = TextLoader(file_path, encoding="utf-8")
documents = loader.load()
print(f"✓ Loaded {len(documents)} document(s)")

# Create chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50)
chunks = text_splitter.split_documents(documents=documents)
print(f"✓ Created {len(chunks)} chunk(s)")

# Create embeddings
load_dotenv()
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=1024)

# Create vector store
vector_store = FAISS.from_documents(documents=chunks, embedding=embedding_model)
print("✓ Vector store created")

✓ Loaded 1 document(s)
✓ Created 2 chunk(s)
✓ Vector store created


### Hybrid Search (BM25 + Vector)

In [3]:
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever

# Create BM25 retriever (keyword search)
bm25_retriever = BM25Retriever.from_documents(chunks)

# Create vector retriever (semantic search)
vector_retriever = vector_store.as_retriever(search_kwargs={"k": 4})

# Combine both retrievers
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.5, 0.5]
)

print("✓ Hybrid retriever created (BM25 + Vector with equal weights)")

ModuleNotFoundError: No module named 'langchain.retrievers'

In [ ]:
# Test hybrid search
query = "story of Captain Ahab"
results = hybrid_retriever.invoke(query)

print(f"Query: '{query}'")
print(f"\nFound {len(results)} relevant document(s):\n")

for i, doc in enumerate(results, 1):
    print(f"{i}. {doc.page_content[:100]}...\n")

### Adjust Weights

In [ ]:
# Try different weights
custom_hybrid = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.3, 0.7]  # More weight to semantic search
)

query = "fantasy adventure"
results = custom_hybrid.invoke(query)

print(f"Query (weights 0.3 BM25, 0.7 Vector): '{query}'")
print(f"Results: {len(results)} document(s)")

### Use Hybrid Results in RAG Chain

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

# Format documents for context
def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

# Create RAG chain
prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer based on this context:\n{context}"),
    ("user", "{question}")
])

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
output_parser = StrOutputParser()

rag_chain = prompt | llm | output_parser

# Get hybrid search results and pass to RAG chain
query = "What is Moby Dick about?"
context_docs = hybrid_retriever.invoke(query)
context = format_docs(context_docs)

answer = rag_chain.invoke({
    "context": context,
    "question": query
})

print(f"Question: {query}\n")
print(f"Answer: {answer}")